In [2]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [5]:
train_df = pd.read_csv("/content/train_df.csv")
test_df = pd.read_csv("/content/test_df.csv")

In [9]:
TARGET = "readmitted"

In [7]:
train_df.head()

,age,gender,primary_diagnosis,num_procedures,days_in_hospital,comorbidity_score,discharge_to,readmitted
0,69,Male,Heart Disease,1,2,1,Home Health Care,0
1,32,Female,COPD,2,13,2,Rehabilitation Facility,0
2,89,Male,Diabetes,1,7,1,Home,0
3,78,Male,COPD,9,2,2,Skilled Nursing Facility,0
4,38,Male,Diabetes,6,4,4,Rehabilitation Facility,0


In [10]:
test_df.head()

,age,gender,primary_diagnosis,num_procedures,days_in_hospital,comorbidity_score,discharge_to
0,52,Male,Heart Disease,3,9,3,Home
1,47,Female,Diabetes,2,4,0,Skilled Nursing Facility
2,72,Female,Heart Disease,7,12,4,Home
3,18,Female,COPD,5,14,3,Home
4,32,Male,Heart Disease,9,2,4,Rehabilitation Facility


In [11]:
train_df = train_df.drop_duplicates().copy()

num_cols = train_df.select_dtypes(include="number").columns.tolist()
cat_cols = train_df.select_dtypes(include="object").columns.tolist()
if TARGET in num_cols:
  num_cols.remove(TARGET)

for col in num_cols:
  train_df[col] = train_df[col].fillna(train_df[col].median())
for col in cat_cols:
  if not train_df[col].mode().empty:
    train_df[col] = train_df[col].fillna(train_df[col].mode()[0])


In [12]:

y_full = train_df[TARGET].copy()
train_features = train_df.drop(columns=[TARGET]).copy()
test_features = test_df.copy()

train_features["__is_train__"] = 1
test_features["__is_train__"] = 0

combined = pd.concat([train_features, test_features], axis=0, ignore_index=True)
cat_features = combined.select_dtypes(include="object").columns.tolist()
combined_encoded = pd.get_dummies(combined, columns=cat_features, drop_first=True)

X_full = combined_encoded[combined_encoded["__is_train__"] == 1].drop(
    columns=["__is_train__"]
)
X_submission = combined_encoded[combined_encoded["__is_train__"] == 0].drop(
    columns=["__is_train__"]
)
display(X_full.head(3))

,age,num_procedures,days_in_hospital,comorbidity_score,gender_Male,primary_diagnosis_Diabetes,primary_diagnosis_Heart Disease,primary_diagnosis_Hypertension,primary_diagnosis_Kidney Disease,discharge_to_Home Health Care,discharge_to_Rehabilitation Facility,discharge_to_Skilled Nursing Facility
0,69,1,2,1,True,False,True,False,False,True,False,False
1,32,2,13,2,False,False,False,False,False,False,True,False
2,89,1,7,1,True,True,False,False,False,False,False,False


In [13]:

X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.20, random_state=42, stratify=y_full
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)


model = LogisticRegression(penalty="l2", C=1.0, class_weight="balanced", max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [14]:
y_pred = model.predict(X_val_scaled)
y_prob = model.predict_proba(X_val_scaled)[:, 1]

auc = roc_auc_score(y_val, y_prob)
tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

print(f"\n--- Model Evaluation ---")
print(f"ROC-AUC Score: {auc:.4f}\n")
print(f"True Negatives : {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives : {tp}\n")



--- Model Evaluation ---
ROC-AUC Score: 0.5037

True Negatives : 417
False Positives: 395
False Negatives: 91
True Positives : 97



In [15]:

print("--- Clinical Cost Discussion ---")
print(
    "False Negatives (FN): High Cost. A high-risk patient is falsely identified"
    " as safe, sending them home without post-discharge care and increasing"
    " risk of severe complications or emergency readmission."
)
print(
    "False Positives (FP): Low Cost. A low-risk patient is flagged as"
    " high-risk, leading to unnecessary follow-up calls or care coordination"
    " expenses, but ensuring patient safety."
)
print(
    "Conclusion: In clinical settings, minimizing False Negatives (maximizing"
    " Recall) is prioritized over minimizing False Positives.\n"
)

--- Clinical Cost Discussion ---
False Negatives (FN): High Cost. A high-risk patient is falsely identified as safe, sending them home without post-discharge care and increasing risk of severe complications or emergency readmission.
False Positives (FP): Low Cost. A low-risk patient is flagged as high-risk, leading to unnecessary follow-up calls or care coordination expenses, but ensuring patient safety.
Conclusion: In clinical settings, minimizing False Negatives (maximizing Recall) is prioritized over minimizing False Positives.



In [16]:
X_submission_scaled = scaler.transform(X_submission)
submission_pred = model.predict(X_submission_scaled)
submission_prob = model.predict_proba(X_submission_scaled)[:, 1]

submission = pd.DataFrame({
    "patient_id": range(1, len(test_df) + 1),
    "readmitted_prediction": submission_pred,
    "readmission_probability": submission_prob.round(4),
})

submission.to_csv("submission.csv", index=False)
files.download("submission.csv")
print("submission.csv generated and downloaded!")

display(submission.head(10))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

submission.csv generated and downloaded!


,patient_id,readmitted_prediction,readmission_probability
0,1,0,0.4843
1,2,1,0.5077
2,3,0,0.4919
3,4,0,0.4780
4,5,0,0.4694
5,6,1,0.5020
6,7,0,0.4726
7,8,1,0.5049
8,9,1,0.5004
9,10,1,0.5013
